In [1]:
import pandas as pd
from Levenshtein import distance
import numpy as np
import re
from collections import defaultdict

In [2]:
df_csv = pd.read_excel('data_new/2025_06_03_Liste der Entschädigungsämter mit Adressen, Varianten, Normalisierungen und Archiven.xlsx')
df_csv.head()

,Layout class,Filename,Bundesland,Transkription Compensation Office,Normalisierung Compensation Office1,Normalisierung Compensation Office2,Zuständiges Archiv,Zuständiges Archiv - GND-Nummer,Unnamed: 8
0,ABC-Karten,1907_12_31_115_0,Niedersachsen,Goslar,Goslar,NaN,Niedersächsisches Landesarchiv,6510041-4,NaN
1,ABC-Karten,1909_01_06_66_0,Niedersachsen,Goslar. Stadt,Goslar,NaN,Niedersächsisches Landesarchiv,6510041-4,NaN
2,ABC-Karten,1904_01_08_54_0,Niedersachsen,Göttingen-Stadt,Göttingen,NaN,Niedersächsisches Landesarchiv,6510041-4,NaN
3,ABC-Karten,1909_01_04_67_0,Niedersachsen,Hannover,Hannover,NaN,Niedersächsisches Landesarchiv,6510041-4,NaN
4,ABC-Karten,1907_09_13_66_0,Niedersachsen,Hannover - Stadt,Hannover,NaN,Niedersächsisches Landesarchiv,6510041-4,NaN


In [3]:
df_json = pd.read_json('compoff_extracted.jsonl', lines=True)
df_json.head()

,CompensationOffice1,BZKNr,filename
0,Regierungsbezirksamt für Wiedergutmachung und ...,65477,1875_00_00_101_0.jpg
1,Landesamt für die Wiedergutmachung Karlsruhe,EK 20093 A,1875_04_05_16_0.jpg
2,Niedersachsen,1 27574 c,1875_06_15_14_0.jpg
3,Bezirksamt für Wiedergutmachung Neustadt a.d.W.,200 488,1875_04_15_1_0.jpg
4,Entschädigungsamt Berlin,70 002,1875_07_12_9_0.jpg


In [4]:
print(len(df_csv))
print(len(df_json))

n = df_json['CompensationOffice1'].isin(df_csv['Transkription Compensation Office ']).sum()
print(f'Number of matches: {n}')

432
1901284
Number of matches: 1121989


In [5]:
print(len(df_csv))
print(len(df_json))

n = df_json['CompensationOffice1'].isin(df_csv['Normalisierung Compensation Office1']).sum()
print(f'Number of matches: {n}')

432
1901284
Number of matches: 1271792


# Matching based on Lvenshtein Distance

In [6]:
compensation_offices = list(df_csv['Transkription Compensation Office '])

# remove linebreaks and sequences of whitespaces
def remove_linebreak(s):
    s = str(s).replace('\n', ' ').replace('\r', ' ')
    return re.sub(r'\s+', ' ', s).strip().lower()

# return the minimum distance and the best match up to a threshold of half the length of the cleaned name
def get_min_distance_dynamic_normalized(name):
    name_clean = remove_linebreak(name)
    threshold = len(name_clean) // 2
    
    min_dist = float('inf')
    best_match = None
    for office in compensation_offices:
        office_clean = remove_linebreak(office)
        d = distance(name_clean, office_clean)
        if d == 0:
            return 0, office
        if d < min_dist:
            min_dist = d
            best_match = office
    if min_dist <= threshold:
        return min_dist, best_match
    return np.nan, None

results = df_json['CompensationOffice1'].apply(get_min_distance_dynamic_normalized)

df_matches = pd.DataFrame({
    'CompensationOffice1': df_json['CompensationOffice1'].apply(remove_linebreak),
    'MatchedOffice': results.apply(lambda x: x[1]),
    'MatchDistance': results.apply(lambda x: x[0])
})


df_matches.head(20)

,CompensationOffice1,MatchedOffice,MatchDistance
0,regierungsbezirksamt für wiedergutmachung und ...,Regierungsbezirksamt \nfür Wiedergutmachung \n...,0.0
1,landesamt für die wiedergutmachung karlsruhe,Landesamt für die Wiedergutmachung \nKarlsruhe,0.0
2,niedersachsen,Niedersachsen,0.0
3,bezirksamt für wiedergutmachung neustadt a.d.w.,Bezirksamt \nfür Wiedergutmachung \nNeustadt a...,2.0
4,entschädigungsamt berlin,Entschädigungsamt Berlin,0.0
5,entschädigungsamt berlin,Entschädigungsamt Berlin,0.0
6,köln,Köln,0.0
7,amt für wiedergutmachung des selbstkantkreises...,Amt für Wiedergutmachung des Selfkantkreises G...,9.0
8,landesentschädigungsamt schleswig-holstein in ...,Landesentschädigungsamt\nSchleswig-Holstein in...,0.0
9,hessen kassel,Hessen Kassel,0.0


In [7]:
print(f"Max MatchDistance: {df_matches['MatchDistance'].max()}")

Max MatchDistance: 67.0


In [8]:
total = len(df_matches)
max_dist = int(df_matches['MatchDistance'].max()) + 1
for i in range(max_dist):
    count = (df_matches['MatchDistance'] == i).sum()

    pct = count / total * 100    
    print(f'MatchDistance = {i}: {count} ({pct:.2f}%)')

MatchDistance = 0: 1422553 (74.82%)
MatchDistance = 1: 64181 (3.38%)
MatchDistance = 2: 24338 (1.28%)
MatchDistance = 3: 22174 (1.17%)
MatchDistance = 4: 43427 (2.28%)
MatchDistance = 5: 10556 (0.56%)
MatchDistance = 6: 24863 (1.31%)
MatchDistance = 7: 6000 (0.32%)
MatchDistance = 8: 5527 (0.29%)
MatchDistance = 9: 21281 (1.12%)
MatchDistance = 10: 4723 (0.25%)
MatchDistance = 11: 4948 (0.26%)
MatchDistance = 12: 10445 (0.55%)
MatchDistance = 13: 3799 (0.20%)
MatchDistance = 14: 2865 (0.15%)
MatchDistance = 15: 2546 (0.13%)
MatchDistance = 16: 2157 (0.11%)
MatchDistance = 17: 2515 (0.13%)
MatchDistance = 18: 2200 (0.12%)
MatchDistance = 19: 5012 (0.26%)
MatchDistance = 20: 1370 (0.07%)
MatchDistance = 21: 1184 (0.06%)
MatchDistance = 22: 13140 (0.69%)
MatchDistance = 23: 1557 (0.08%)
MatchDistance = 24: 17755 (0.93%)
MatchDistance = 25: 808 (0.04%)
MatchDistance = 26: 2887 (0.15%)
MatchDistance = 27: 408 (0.02%)
MatchDistance = 28: 279 (0.01%)
MatchDistance = 29: 705 (0.04%)
MatchDista

In [9]:
df_nomatches = df_matches[(df_matches['MatchDistance'] > 0) | (df_matches['MatchDistance'].isna())]

df_nomatches.sort_values('MatchDistance', ascending=True).to_excel('data_new/compensation_office_nomatches.xlsx', index=False)

KeyboardInterrupt: 

In [10]:
df_exact_matches_or_nan = df_matches[(df_matches['MatchDistance'] == 0) | (df_matches['CompensationOffice1'].isna()) | (df_matches['CompensationOffice1'] == 'nan')]

df_exact_matches_or_nan

,CompensationOffice1,MatchedOffice,MatchDistance
0,regierungsbezirksamt für wiedergutmachung und ...,Regierungsbezirksamt \nfür Wiedergutmachung \n...,0.0
1,landesamt für die wiedergutmachung karlsruhe,Landesamt für die Wiedergutmachung \nKarlsruhe,0.0
2,niedersachsen,Niedersachsen,0.0
4,entschädigungsamt berlin,Entschädigungsamt Berlin,0.0
5,entschädigungsamt berlin,Entschädigungsamt Berlin,0.0
...,...,...,...
1901279,bayerisches landesentschädigungsamt,Bayerisches Landesentschädigungsamt,0.0
1901280,entschädigungsamt berlin,Entschädigungsamt Berlin,0.0
1901281,entschädigungsamt berlin,Entschädigungsamt Berlin,0.0
1901282,bezirksamt für wiedergutmachung koblenz,Bezirksamt für Wiedergutmachung Koblenz,0.0


In [11]:
df_json["norm_office"] = df_json["CompensationOffice1"].apply(remove_linebreak)

exact_norm = set(df_exact_matches_or_nan["CompensationOffice1"].dropna())

df_json_filtered = df_json[~df_json["norm_office"].isin(exact_norm)].copy()

print(len(df_json), len(df_json_filtered))
df_json_filtered.head()

1901284 403393


,CompensationOffice1,BZKNr,filename,norm_office
3,Bezirksamt für Wiedergutmachung Neustadt a.d.W.,200 488,1875_04_15_1_0.jpg,bezirksamt für wiedergutmachung neustadt a.d.w.
7,Amt für Wiedergutmachung des Selbstkantkreises...,48 583,1875_12_26_5_0.jpg,amt für wiedergutmachung des selbstkantkreises...
12,Bezirksamt für Wiedergutmachung Köln,407384,1875_00_00_11_0.jpg,bezirksamt für wiedergutmachung köln
13,Der Regierungspräsidium Düsseldorf,78872,1875_06_17_4_0.jpg,der regierungspräsidium düsseldorf
16,"Hansestadt Hamburg, Sozialbehörde, Amt für Wie...",21806 Keb,1875_09_14_7_0.jpg,"hansestadt hamburg, sozialbehörde, amt für wie..."


In [12]:
df_json_filtered.to_json('data_new/compoff_extracted_filtered.jsonl', orient='records', lines=True)

# Continuing with Partial Dataset

In [26]:
df_json_filtered = pd.read_json('data_new/compoff_extracted_filtered.jsonl', lines=True)

df_json_filtered

,CompensationOffice1,BZKNr,filename,norm_office
0,Bezirksamt für Wiedergutmachung Neustadt a.d.W.,200 488,1875_04_15_1_0.jpg,bezirksamt für wiedergutmachung neustadt a.d.w.
1,Amt für Wiedergutmachung des Selbstkantkreises...,48 583,1875_12_26_5_0.jpg,amt für wiedergutmachung des selbstkantkreises...
2,Bezirksamt für Wiedergutmachung Köln,407384,1875_00_00_11_0.jpg,bezirksamt für wiedergutmachung köln
3,Der Regierungspräsidium Düsseldorf,78872,1875_06_17_4_0.jpg,der regierungspräsidium düsseldorf
4,"Hansestadt Hamburg, Sozialbehörde, Amt für Wie...",21806 Keb,1875_09_14_7_0.jpg,"hansestadt hamburg, sozialbehörde, amt für wie..."
...,...,...,...,...
403388,Aachen,41 952,1917_03_27_67_0.jpg,aachen
403389,Bayrisches Landesamt für Verfolgtenwesen,123312/VIII/55,1917_06_05_91_0.jpg,bayrisches landesamt für verfolgtenwesen
403390,Reg.-Präsident / Präsident des Verw.Bezirks,1 32601 a,1917_05_24_21_0.jpg,reg.-präsident / präsident des verw.bezirks
403391,Bezirksamt für Wiedergutmachung,111 427,1917_03_27_4_0.jpg,bezirksamt für wiedergutmachung


# Matching without Punctuation

In [27]:
def remove_linebreak(s):
    s = str(s).replace('\n', ' ').replace('\r', ' ')
    return re.sub(r'\s+', ' ', s).strip().lower()

# remove linebreaks, punctuation and sequences of whitespaces
def remove_punctuation(s):
    p = ['?', '!', '.', ',', ';', ':', '-', '_', '(', ')', '[', ']', '{', '}', '"', "'", '/']
    for char in p:
        s = s.replace(char, ' ')
    return re.sub(r'\s+', ' ', s).strip().lower()

compensation_offices = list(df_csv['Transkription Compensation Office '])

print(compensation_offices[:10])

compensation_offices = [remove_linebreak(remove_punctuation(str(office))) for office in compensation_offices]

print(compensation_offices[:10])

['Goslar', 'Goslar. Stadt', 'Göttingen-Stadt', 'Hannover', 'Hannover - Stadt', 'Helmstedt', 'Hildesheim-Stadt', 'Kreissonderhilfsausschuß \nDelmenhorst', 'Kreissonderhilfsausschuß \ndes Landkreises Grafsch. Schaumburg', 'Kreissonderhilfsausschuß\nLeer/Ostfrld.']
['goslar', 'goslar stadt', 'göttingen stadt', 'hannover', 'hannover stadt', 'helmstedt', 'hildesheim stadt', 'kreissonderhilfsausschuß delmenhorst', 'kreissonderhilfsausschuß des landkreises grafsch schaumburg', 'kreissonderhilfsausschuß leer ostfrld']


In [28]:

#updated version of get_min_distance_dynamic_normalized that also removes punctuation
def get_min_distance_dynamic_normalized(name):
    name_clean = remove_linebreak(remove_punctuation(name))
    threshold = len(name_clean) // 2
    
    min_dist = float('inf')
    best_match = None
    for office in compensation_offices:
        d = distance(name_clean, office)
        if d == 0:
            return 0, office
        if d < min_dist:
            min_dist = d
            best_match = office
    if min_dist <= threshold:
        return min_dist, best_match
    return np.nan, None

results = df_json_filtered['CompensationOffice1'].apply(get_min_distance_dynamic_normalized)

df_matches = pd.DataFrame({
    'CompensationOffice1': df_json_filtered['CompensationOffice1'].apply(remove_punctuation),
    'MatchedOffice': results.apply(lambda x: x[1]),
    'MatchDistance': results.apply(lambda x: x[0])
})

df_matches.head(20)

,CompensationOffice1,MatchedOffice,MatchDistance
0,bezirksamt für wiedergutmachung neustadt a d w,bezirksamt für wiedergutmachung neustadt a d w,0.0
1,amt für wiedergutmachung des selbstkantkreises...,amt für wiedergutmachung des selfkantkreises g...,7.0
2,bezirksamt für wiedergutmachung köln,bezirksamt für wiedergutmachung mainz,4.0
3,der regierungspräsidium düsseldorf,der regierungspräsident düsseldorf,3.0
4,hansestadt hamburg sozialbehörde amt für wiede...,hansestadt hamburg sozialbehörde amt für wiede...,1.0
5,regierungsbezirk für wiedergutmachung und verw...,regierungsbezirksamt für wiedergutmachung und ...,4.0
6,hess staatsministerium,NaN,NaN
7,regierungsbezirksamt für wiedergutmachung und ...,regierungsbezirksamt für wiedergutmachung und ...,2.0
8,freie und hansestadt hamburg,NaN,NaN
9,hess staatsministerium,NaN,NaN


In [29]:
total = len(df_matches)
max_dist = int(df_matches['MatchDistance'].max()) + 1
for i in range(max_dist):
    count = (df_matches['MatchDistance'] == i).sum()

    pct = count / total * 100    
    print(f'MatchDistance = {i}: {count} ({pct:.2f}%)')

MatchDistance = 0: 52695 (13.06%)
MatchDistance = 1: 33758 (8.37%)
MatchDistance = 2: 10533 (2.61%)
MatchDistance = 3: 19465 (4.83%)
MatchDistance = 4: 43230 (10.72%)
MatchDistance = 5: 9382 (2.33%)
MatchDistance = 6: 23238 (5.76%)
MatchDistance = 7: 5665 (1.40%)
MatchDistance = 8: 5928 (1.47%)
MatchDistance = 9: 22167 (5.50%)
MatchDistance = 10: 3739 (0.93%)
MatchDistance = 11: 3571 (0.89%)
MatchDistance = 12: 10493 (2.60%)
MatchDistance = 13: 3929 (0.97%)
MatchDistance = 14: 3367 (0.83%)
MatchDistance = 15: 2666 (0.66%)
MatchDistance = 16: 2449 (0.61%)
MatchDistance = 17: 2543 (0.63%)
MatchDistance = 18: 2401 (0.60%)
MatchDistance = 19: 15760 (3.91%)
MatchDistance = 20: 2480 (0.61%)
MatchDistance = 21: 1545 (0.38%)
MatchDistance = 22: 1307 (0.32%)
MatchDistance = 23: 20327 (5.04%)
MatchDistance = 24: 1531 (0.38%)
MatchDistance = 25: 543 (0.13%)
MatchDistance = 26: 244 (0.06%)
MatchDistance = 27: 601 (0.15%)
MatchDistance = 28: 925 (0.23%)
MatchDistance = 29: 1739 (0.43%)
MatchDistanc

In [30]:
df_exact_matches = df_matches[df_matches['MatchDistance'] == 0]

df_exact_matches 

,CompensationOffice1,MatchedOffice,MatchDistance
0,bezirksamt für wiedergutmachung neustadt a d w,bezirksamt für wiedergutmachung neustadt a d w,0.0
11,regierungsbezirksamt für wiedergutmachung und ...,regierungsbezirksamt für wiedergutmachung und ...,0.0
25,regierungsbezirksamt für wiedergutmachung und ...,regierungsbezirksamt für wiedergutmachung und ...,0.0
37,regierungsbezirksamt für wiedergutmachung und ...,regierungsbezirksamt für wiedergutmachung und ...,0.0
39,landesamt für die wiedergutmachung tübingen,landesamt für die wiedergutmachung tübingen,0.0
...,...,...,...
403358,oberfinanzdirektion köln,oberfinanzdirektion köln,0.0
403359,regierungsbezirksamt für wiedergutmachung und ...,regierungsbezirksamt für wiedergutmachung und ...,0.0
403366,bezirksamt für wiedergutmachung neustadt a d w,bezirksamt für wiedergutmachung neustadt a d w,0.0
403376,oberfinanzdirektion köln,oberfinanzdirektion köln,0.0


In [ ]:
df_json_filtered["norm_office"] = df_json_filtered["norm_office"].apply(remove_punctuation)

exact_norm = set(df_exact_matches["CompensationOffice1"].dropna())

df_json_filtered = df_json_filtered[~df_json_filtered["norm_office"].isin(exact_norm)].copy()

df_json_filtered

,CompensationOffice1,BZKNr,filename,norm_office
1,Amt für Wiedergutmachung des Selbstkantkreises...,48 583,1875_12_26_5_0.jpg,amt für wiedergutmachung des selbstkantkreises...
2,Bezirksamt für Wiedergutmachung Köln,407384,1875_00_00_11_0.jpg,bezirksamt für wiedergutmachung köln
3,Der Regierungspräsidium Düsseldorf,78872,1875_06_17_4_0.jpg,der regierungspräsidium düsseldorf
4,"Hansestadt Hamburg, Sozialbehörde, Amt für Wie...",21806 Keb,1875_09_14_7_0.jpg,hansestadt hamburg sozialbehörde amt für wiede...
5,Regierungsbezirk für Wiedergutmachung und verw...,47 691,1875_04_05_13_0.jpg,regierungsbezirk für wiedergutmachung und verw...
...,...,...,...,...
403387,Reg.-Präsident /Präsident des Vereinsbezirks H...,1 27101 a,1917_06_23_32_0.jpg,reg präsident präsident des vereinsbezirks han...
403388,Aachen,41 952,1917_03_27_67_0.jpg,aachen
403389,Bayrisches Landesamt für Verfolgtenwesen,123312/VIII/55,1917_06_05_91_0.jpg,bayrisches landesamt für verfolgtenwesen
403390,Reg.-Präsident / Präsident des Verw.Bezirks,1 32601 a,1917_05_24_21_0.jpg,reg präsident präsident des verw bezirks


In [32]:
df_no_matches_no_punctuation = df_matches[(df_matches['MatchDistance'] > 0) | (df_matches['MatchDistance'].isna())]
df_no_matches_no_punctuation = df_no_matches_no_punctuation.sort_values('MatchDistance', ascending=False)
df_no_matches_no_punctuation.head(20)

,CompensationOffice1,MatchedOffice,MatchDistance
89799,der regierungspräsident dezernat für wiedergut...,der regierungspräsident dezernat für wiedergut...,64.0
10300,freie und hansestadt hamburg behörde für arbei...,behörde für arbeit gesunheit und soziales amt ...,61.0
244488,der regierungspräsident dezernat für wiedergut...,der regierungspräsident dezernat für wiedergut...,60.0
388722,der regierungspräsident dezernat für wiedergut...,der regierungspräsident dezernat für wiedergut...,60.0
62861,freie und hansestadt hamburg behörde für arbei...,behörde für arbeit gesunheit und soziales amt ...,60.0
245394,der regierungspräsident dezernat für wiedergut...,der regierungspräsident dezernat für wiedergut...,60.0
399515,freie und hansestadt hamburg behörde für arbei...,behörde für arbeit gesunheit und soziales amt ...,60.0
242288,der regierungspräsident – dezernat für wiederg...,der regierungspräsident dezernat für wiedergut...,60.0
345444,freie und hansestadt hamburg behörde für arbei...,behörde für arbeit gesunheit und soziales amt ...,60.0
48966,regierungsbezirksamt für wiedergutmachung und ...,regierungsbezirksamt für wiedergutmachung und ...,60.0


In [ ]:
df_no_matches_no_punctuation.to_excel('data_new/compensation_office_nomatches_no_punctuation.xlsx', index=False)

# No Matches

In [33]:
no_matched_office_df = df_nomatches[df_nomatches['MatchDistance'].isna()]

no_matched_office_df

,CompensationOffice1,MatchedOffice,MatchDistance
21,hess. staatsministerium,NaN,NaN
30,nan,NaN,NaN
32,freie und hansestadt hamburg,NaN,NaN
34,hess. staatsministerium,NaN,NaN
40,nan,NaN,NaN
...,...,...,...
1901202,hng,NaN,NaN
1901238,hng,NaN,NaN
1901254,reg.-präsident /präsident des vereinsbezirks h...,NaN,NaN
1901262,bayrisches landesamt für verfolgtenwesen,NaN,NaN


In [34]:
no_matched_office_df_set = set(no_matched_office_df['CompensationOffice1'])

len(no_matched_office_df_set)
for office in no_matched_office_df_set:
    print(office)


ksha lütt
reg.-präsident / präsident des v. bezirks hannover
bayrisches amt
niedersachsen / reg.-präsident / präsident des verw.-bezirks - hannover
ra adlerstein, d-dorf
bhf-nb3-01481
bay. landesamt für die opfer des nationalsozialismus
reg.-präsident / präsidentiales verwaltungsbeamt
kreissozialamt wattenstedt-salzgitter
st. h. 1665/1966 h
rheinisch-westfälische
niedersachsen-präsidium / präsidenten des merkwaertes
reparatur prüfungsamt (wiesbaden)
rostock
bundeszentrale für die opfer des nationalsozialismus
st.h. 602 s
erlanger
freib
kreissozialamt schauemburg-lippe
regierungsbezirk münz
köln, riehler pl.2, 5090 köln
kreisverwaltung arnswald (mf.)
x8lm
united restitution office, frankfurt (m)
ksh bameln-parmont
d1 380 e 2081 c 380
saarburg bezirksamt
united restitution office frankfurt/m. grünburgweg 119
st. h. 2018/1290 m
haftsschuld
landeszentrale - registrazentrale
amt für wiedergutmachung des bundesministeriums für vertriebene, flüchtlinge und kriegsbeschädigte in berlin
bah
reg

# Swapping out single Tokens with a match distance of 1

In [35]:
def office_token_set(series):
    tokens = set()
    cleaned = (
        series.fillna("")
        .astype(str)
        .str.lower()
        .str.replace(r"\n", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    for name in cleaned:
        tokens.update(token for token in re.findall(r"\b\w+\b", name, flags=re.UNICODE)
            if not token.isdigit())
    return tokens

csv_tokens = office_token_set(df_csv["Transkription Compensation Office "].dropna().apply(remove_punctuation).apply(remove_linebreak))
json_tokens = office_token_set(df_json_filtered["norm_office"])

print(f"csv_tokens: {len(csv_tokens)}")
print(f"json_tokens: {len(json_tokens)}")

csv_tokens: 359
json_tokens: 10692


In [36]:
for t in csv_tokens:
    if len(t) == 3:
        print(t)

dez
ist
ems
und
amt
ofd
reg
süd
etc
das
lrb
hdt
vb4
str
der
fin
des
min
die
für
abt
neu
bad


In [37]:
csv_tokens_by_len = defaultdict(list)
for token in csv_tokens:
    if len(token) >= 3:
        csv_tokens_by_len[len(token)].append(token)

json_to_csv_distance1_map = {}
for json_token in json_tokens:
    if json_token in csv_tokens:
        continue

    candidate_tokens = (
        csv_tokens_by_len[len(json_token) - 1]
        + csv_tokens_by_len[len(json_token)]
        + csv_tokens_by_len[len(json_token) + 1]
    )

    matches_dist1 = [csv_token for csv_token in candidate_tokens if distance(json_token, csv_token) == 1]
    if matches_dist1:
        json_to_csv_distance1_map[json_token] = sorted(set(matches_dist1))

single_token_matches_dist1_len3 =sorted(
    [item for item in json_to_csv_distance1_map.items() if len(item[1]) == 1],
    key=lambda item: len(item[0])
)

print(len(single_token_matches_dist1_len3))
single_token_matches_dist1_len3

1004


[('re', ['reg']),
 ('eg', ['reg']),
 ('lr', ['lrb']),
 ('b4', ['vb4']),
 ('dr', ['der']),
 ('fr', ['für']),
 ('tr', ['str']),
 ('da', ['das']),
 ('di', ['die']),
 ('er', ['der']),
 ('ad', ['bad']),
 ('dt', ['hdt']),
 ('vb', ['vb4']),
 ('ba', ['bad']),
 ('of', ['ofd']),
 ('bd', ['bad']),
 ('ht', ['hdt']),
 ('hd', ['hdt']),
 ('dz', ['dez']),
 ('as', ['das']),
 ('mt', ['amt']),
 ('mi', ['min']),
 ('sr', ['str']),
 ('rg', ['reg']),
 ('rb', ['lrb']),
 ('nu', ['neu']),
 ('bah', ['bad']),
 ('rad', ['bad']),
 ('brd', ['bad']),
 ('stg', ['str']),
 ('ddr', ['der']),
 ('bav', ['bad']),
 ('lab', ['lrb']),
 ('bes', ['des']),
 ('old', ['ofd']),
 ('bay', ['bad']),
 ('lüd', ['süd']),
 ('ojd', ['ofd']),
 ('rbg', ['reg']),
 ('hss', ['hess']),
 ('xes', ['des']),
 ('vc4', ['vb4']),
 ('per', ['der']),
 ('her', ['der']),
 ('lre', ['lrb']),
 ('lro', ['lrb']),
 ('erb', ['lrb']),
 ('ref', ['reg']),
 ('oed', ['ofd']),
 ('rog', ['reg']),
 ('ung', ['und']),
 ('opd', ['ofd']),
 ('diö', ['die']),
 ('dat', ['das']),

In [38]:
unique_distance1_token_map = {
    json_token: csv_candidates[0]
    for json_token, csv_candidates in json_to_csv_distance1_map.items()
    if len(csv_candidates) == 1
}

def replace_unique_distance1_tokens(text):
    text = "" if pd.isna(text) else str(text)
    return re.sub(
        r"\b\w+\b",
        lambda match: unique_distance1_token_map.get(match.group(0).lower(), match.group(0)),
        text,
        flags=re.UNICODE
    )

df_json_filtered['CompensationOffice_token_swap'] = df_json_filtered['norm_office'].apply(replace_unique_distance1_tokens)

print(f"Unique token replacements available: {len(unique_distance1_token_map)}")

results = df_json_filtered['CompensationOffice_token_swap'].apply(get_min_distance_dynamic_normalized)
df_matches_token_swap = pd.DataFrame({
    'CompensationOffice1': df_json_filtered['norm_office'],
    'MatchedOffice': results.apply(lambda x: x[1]),
    'MatchDistance': results.apply(lambda x: x[0])
})

df_matches_token_swap.head(20)

Unique token replacements available: 1004


,CompensationOffice1,MatchedOffice,MatchDistance
1,amt für wiedergutmachung des selbstkantkreises...,amt für wiedergutmachung des selfkantkreises g...,7.0
2,bezirksamt für wiedergutmachung köln,bezirksamt für wiedergutmachung mainz,4.0
3,der regierungspräsidium düsseldorf,der regierungspräsident düsseldorf,3.0
4,hansestadt hamburg sozialbehörde amt für wiede...,hansestadt hamburg sozialbehörde amt für wiede...,1.0
5,regierungsbezirk für wiedergutmachung und verw...,regierungsbezirksamt für wiedergutmachung und ...,3.0
6,hess staatsministerium,NaN,NaN
7,regierungsbezirksamt für wiedergutmachung und ...,regierungsbezirksamt für wiedergutmachung und ...,2.0
8,freie und hansestadt hamburg,NaN,NaN
9,hess staatsministerium,NaN,NaN
10,hansestadt hamburg sozialbehörde amt für wiede...,hansestadt hamburg sozialbehörde amt für wiede...,1.0


In [39]:
total = len(df_matches_token_swap)
max_dist = int(df_matches_token_swap['MatchDistance'].max()) + 1
for i in range(max_dist):
    count = (df_matches_token_swap['MatchDistance'] == i).sum()

    pct = count / total * 100    
    print(f'MatchDistance = {i}: {count} ({pct:.2f}%)')

MatchDistance = 0: 12773 (3.64%)
MatchDistance = 1: 21254 (6.06%)
MatchDistance = 2: 11019 (3.14%)
MatchDistance = 3: 20811 (5.93%)
MatchDistance = 4: 41667 (11.88%)
MatchDistance = 5: 9178 (2.62%)
MatchDistance = 6: 23891 (6.81%)
MatchDistance = 7: 4713 (1.34%)
MatchDistance = 8: 6043 (1.72%)
MatchDistance = 9: 22343 (6.37%)
MatchDistance = 10: 3564 (1.02%)
MatchDistance = 11: 3458 (0.99%)
MatchDistance = 12: 10973 (3.13%)
MatchDistance = 13: 3680 (1.05%)
MatchDistance = 14: 3331 (0.95%)
MatchDistance = 15: 2789 (0.80%)
MatchDistance = 16: 2635 (0.75%)
MatchDistance = 17: 2330 (0.66%)
MatchDistance = 18: 2849 (0.81%)
MatchDistance = 19: 15756 (4.49%)
MatchDistance = 20: 3595 (1.03%)
MatchDistance = 21: 1544 (0.44%)
MatchDistance = 22: 1526 (0.44%)
MatchDistance = 23: 20328 (5.80%)
MatchDistance = 24: 1535 (0.44%)
MatchDistance = 25: 495 (0.14%)
MatchDistance = 26: 260 (0.07%)
MatchDistance = 27: 594 (0.17%)
MatchDistance = 28: 969 (0.28%)
MatchDistance = 29: 1715 (0.49%)
MatchDistance

In [40]:
def token_count(s):
    return len(str(s).split())

extracted_longer = sorted(
    {
        (str(a), str(b))
        for a, b in zip(df_matches_token_swap['CompensationOffice1'], df_matches_token_swap['MatchedOffice'])
        if token_count(a) > token_count(b) and str(b) != 'nan'
    },
    key=lambda item: token_count(item[0])
)

extracted_longer

[('früher koblenz', 'koblenz'),
 ('uro berlin', 'berlin'),
 ('hamburg az', 'hamburg'),
 ('umw köln', 'köln'),
 ('karl wolke', 'karlsruhe'),
 ('münchner eg', 'münchen'),
 ('neu isenburg', 'oldenburg'),
 ('ilva darmstadt', 'darmstadt'),
 ('kreis osnabrück', 'osnabrück'),
 ('bayern münchen', 'münchen'),
 ('ojj münster', 'münster'),
 ('bva hamburg', 'hamburg'),
 ('lvh hamburg', 'hamburg'),
 ('münchen beg', 'münchen'),
 ('iva hamburg', 'hamburg'),
 ('ba oldenburg', 'oldenburg'),
 ('nrw münster', 'münster'),
 ('karlsruhe freiburg', 'karlsruhe'),
 ('münchen 2', 'münchen'),
 ('stadt oldenburg', 'oldenburg'),
 ('braunschweig ii', 'braunschweig'),
 ('os nabruck', 'osnabrück'),
 ('land hessen', 'hessen'),
 ('detmold köln', 'detmold'),
 ('niedersachsen oldenburg', 'niedersachsen'),
 ('köl n', 'köln'),
 ('nürnberg a', 'arnsberg'),
 ('bayer london', 'bayern'),
 ('va hamburg', 'hamburg'),
 ('ksha oldenburg', 'oldenburg'),
 ('ksha bremen', 'bremen'),
 ('oldenburg olb', 'oldenburg'),
 ('neustadt m', 'n

# Swapping out single tokens with a match distance of up to half of the token's lenght

In [41]:
json_to_csv_closest_map = {}
for json_token in json_tokens:
    if json_token in csv_tokens:
        continue
    
    max_distance = len(json_token) // 2
    best_match = None
    best_dist = float('inf')
    
    for csv_token in csv_tokens:
        if len(csv_token) >= 3:
            if abs(len(csv_token) - len(json_token)) > max_distance:
                continue
            dist = distance(json_token, csv_token)
            if dist <= max_distance and dist < best_dist:
                best_dist = dist
                best_match = csv_token
    
    if best_match is not None:
        json_to_csv_closest_map[json_token] = [best_match]

single_token_matches_closest = sorted(
    json_to_csv_closest_map.items(),
    key=lambda item: len(item[0])
)
print(len(single_token_matches_closest))
single_token_matches_closest

7013


[('re', ['reg']),
 ('eg', ['reg']),
 ('lr', ['lrb']),
 ('b4', ['vb4']),
 ('dr', ['der']),
 ('fr', ['für']),
 ('tr', ['str']),
 ('da', ['das']),
 ('di', ['die']),
 ('er', ['der']),
 ('ad', ['bad']),
 ('dt', ['hdt']),
 ('st', ['ist']),
 ('vb', ['vb4']),
 ('ba', ['bad']),
 ('de', ['dez']),
 ('of', ['ofd']),
 ('bd', ['bad']),
 ('ht', ['hdt']),
 ('hd', ['hdt']),
 ('dz', ['dez']),
 ('as', ['das']),
 ('mt', ['amt']),
 ('mi', ['min']),
 ('sr', ['str']),
 ('es', ['ems']),
 ('rg', ['reg']),
 ('rb', ['lrb']),
 ('nu', ['neu']),
 ('bah', ['bad']),
 ('rad', ['bad']),
 ('brd', ['bad']),
 ('stg', ['str']),
 ('ddr', ['der']),
 ('bav', ['bad']),
 ('lab', ['lrb']),
 ('drs', ['das']),
 ('ast', ['ist']),
 ('ain', ['fin']),
 ('bes', ['des']),
 ('old', ['ofd']),
 ('bay', ['bad']),
 ('lüd', ['süd']),
 ('ojd', ['ofd']),
 ('rbg', ['reg']),
 ('del', ['dez']),
 ('hss', ['hess']),
 ('xes', ['des']),
 ('art', ['amt']),
 ('vc4', ['vb4']),
 ('per', ['der']),
 ('dad', ['das']),
 ('her', ['der']),
 ('lre', ['lrb']),
 (

In [42]:
unique_distance1_token_map = {
    json_token: csv_candidates[0]
    for json_token, csv_candidates in json_to_csv_closest_map.items()
    if len(csv_candidates) == 1
}

def replace_unique_distance1_tokens(text):
    text = "" if pd.isna(text) else str(text)
    return re.sub(
        r"\b\w+\b",
        lambda match: unique_distance1_token_map.get(match.group(0).lower(), match.group(0)),
        text,
        flags=re.UNICODE
    )

df_json_filtered['CompensationOffice_token_swap'] = df_json_filtered['norm_office'].apply(replace_unique_distance1_tokens)

print(f"Unique token replacements available: {len(unique_distance1_token_map)}")

results = df_json_filtered['CompensationOffice_token_swap'].apply(get_min_distance_dynamic_normalized)
df_matches_token_swap = pd.DataFrame({
    'CompensationOffice1': df_json_filtered['norm_office'],
    'MatchedOffice': results.apply(lambda x: x[1]),
    'MatchDistance': results.apply(lambda x: x[0])
})

df_matches_token_swap.head(20)

Unique token replacements available: 7013


,CompensationOffice1,MatchedOffice,MatchDistance
1,amt für wiedergutmachung des selbstkantkreises...,amt für wiedergutmachung des selfkantkreises g...,10.0
2,bezirksamt für wiedergutmachung köln,bezirksamt für wiedergutmachung mainz,4.0
3,der regierungspräsidium düsseldorf,der regierungspräsident düsseldorf,3.0
4,hansestadt hamburg sozialbehörde amt für wiede...,hansestadt hamburg sozialbehörde amt für wiede...,1.0
5,regierungsbezirk für wiedergutmachung und verw...,regierungsbezirksamt für wiedergutmachung und ...,3.0
6,hess staatsministerium,NaN,NaN
7,regierungsbezirksamt für wiedergutmachung und ...,regierungsbezirksamt für wiedergutmachung und ...,0.0
8,freie und hansestadt hamburg,NaN,NaN
9,hess staatsministerium,NaN,NaN
10,hansestadt hamburg sozialbehörde amt für wiede...,hansestadt hamburg sozialbehörde amt für wiede...,1.0


In [43]:
total = len(df_matches_token_swap)
max_dist = int(df_matches_token_swap['MatchDistance'].max()) + 1
for i in range(max_dist):
    count = (df_matches_token_swap['MatchDistance'] == i).sum()

    pct = count / total * 100    
    print(f'MatchDistance = {i}: {count} ({pct:.2f}%)')

MatchDistance = 0: 26718 (7.62%)
MatchDistance = 1: 19486 (5.56%)
MatchDistance = 2: 6634 (1.89%)
MatchDistance = 3: 19193 (5.47%)
MatchDistance = 4: 41087 (11.72%)
MatchDistance = 5: 6784 (1.93%)
MatchDistance = 6: 23953 (6.83%)
MatchDistance = 7: 3818 (1.09%)
MatchDistance = 8: 6261 (1.79%)
MatchDistance = 9: 25207 (7.19%)
MatchDistance = 10: 3461 (0.99%)
MatchDistance = 11: 2775 (0.79%)
MatchDistance = 12: 13350 (3.81%)
MatchDistance = 13: 4640 (1.32%)
MatchDistance = 14: 2040 (0.58%)
MatchDistance = 15: 3274 (0.93%)
MatchDistance = 16: 3127 (0.89%)
MatchDistance = 17: 1801 (0.51%)
MatchDistance = 18: 1906 (0.54%)
MatchDistance = 19: 15997 (4.56%)
MatchDistance = 20: 2269 (0.65%)
MatchDistance = 21: 1140 (0.33%)
MatchDistance = 22: 797 (0.23%)
MatchDistance = 23: 19975 (5.70%)
MatchDistance = 24: 1462 (0.42%)
MatchDistance = 25: 350 (0.10%)
MatchDistance = 26: 202 (0.06%)
MatchDistance = 27: 700 (0.20%)
MatchDistance = 28: 1404 (0.40%)
MatchDistance = 29: 1775 (0.51%)
MatchDistance 

In [ ]:
df_leftovers = df_matches_token_swap[df_matches_token_swap['MatchDistance'] == 4]

df_leftovers

,CompensationOffice1,MatchedOffice,MatchDistance
1,amt für wiedergutmachung des selbstkantkreises...,amt für wiedergutmachung des selfkantkreises g...,10.0
2,bezirksamt für wiedergutmachung köln,bezirksamt für wiedergutmachung mainz,4.0
3,der regierungspräsidium düsseldorf,der regierungspräsident düsseldorf,3.0
4,hansestadt hamburg sozialbehörde amt für wiede...,hansestadt hamburg sozialbehörde amt für wiede...,1.0
5,regierungsbezirk für wiedergutmachung und verw...,regierungsbezirksamt für wiedergutmachung und ...,3.0
...,...,...,...
403387,reg präsident präsident des vereinsbezirks han...,der präsident des verwaltungsbezirks in oldenburg,26.0
403388,aachen,münchen,3.0
403389,bayrisches landesamt für verfolgtenwesen,bayerisches landesentschädigungsamt,16.0
403390,reg präsident präsident des verw bezirks,reg präsident hildesheim,20.0


In [46]:
for t in csv_tokens:
    if t == 'regierungspräsidium':
        print(t)

regierungspräsidium


# Calling LLM via API using NaN values

In [ ]:
from groq import Groq
from huggingface_hub import InferenceClient
from huggingface_hub.utils import HfHubHTTPError

import time
import json
import os
from dotenv import load_dotenv
from tqdm import tqdm


In [ ]:
load_dotenv()

LLM_PROVIDER = os.getenv("LLM_PROVIDER", "groq").strip().lower()
HF_MODEL = os.getenv("HF_MODEL", "Qwen/Qwen3-8B")
GROQ_MODEL = os.getenv("GROQ_MODEL", "llama-3.1-8b-instant")

if LLM_PROVIDER == "huggingface":
    HF_API_TOKEN = os.getenv("HF_API_KEY")
    if not HF_API_TOKEN:
        raise ValueError("HF_API_KEY is required when LLM_PROVIDER='huggingface'")
    client = InferenceClient(
        model=HF_MODEL,
        token=HF_API_TOKEN
    )
    ACTIVE_MODEL = HF_MODEL
elif LLM_PROVIDER == "groq":
    GROQ_API_KEY = os.getenv("GROQ_API_KEY")
    if not GROQ_API_KEY:
        raise ValueError("GROQ_API_KEY is required when LLM_PROVIDER='groq'")
    client = Groq(api_key=GROQ_API_KEY)
    ACTIVE_MODEL = GROQ_MODEL
else:
    raise ValueError("LLM_PROVIDER must be either 'huggingface' or 'groq'")

print(f"Using LLM provider: {LLM_PROVIDER}, model: {ACTIVE_MODEL}")

offices_list = "\n".join([f"- {office}" for office in compensation_offices])

In [ ]:

def get_llm_match_batch(query_names, offices_list, max_retries=3):
    
    queries = "\n".join([f"{i+1}. \"{name}\"" for i, name in enumerate(query_names)])
    
    prompt = f"""You are an expert at matching German compensation office names (Entschädigungsämter).

Given these query names:
{queries}

Find the BEST match for EACH from this list:
{offices_list}

Rules:
1. Return ONLY a numbered list with the exact name from the list
2. Consider spelling variations and OCR errors
3. If no match exists, write \"NO_MATCH\"
4. Format: \"1. Matching Name\" or \"1. NO_MATCH\"

Answers:"""

    messages = [{"role": "user", "content": prompt}]
    
    for attempt in range(max_retries):
        try:
            if LLM_PROVIDER == "huggingface":
                response = client.chat_completion(
                    messages=messages,
                    max_tokens=500,
                    temperature=0.1
                )
                text = response.choices[0].message.content
            elif LLM_PROVIDER == "groq":
                response = client.chat.completions.create(
                    model=ACTIVE_MODEL,
                    messages=messages,
                    max_tokens=500,
                    temperature=0.1
                )
                text = response.choices[0].message.content
            else:
                raise ValueError(f"Unsupported LLM provider: {LLM_PROVIDER}")
            
            matches = []
            lines = text.split('\n')
            for line in lines:
                line = line.strip()
                if line and len(line) > 0 and line[0].isdigit():
                    parts = line.split('.', 1) if '.' in line else line.split(')', 1)
                    if len(parts) > 1:
                        match = parts[1].strip()
                        matches.append(match)
            
            while len(matches) < len(query_names):
                matches.append("PARSE_ERROR")
            
            return matches[:len(query_names)]
            
        except HfHubHTTPError as e:
            error_str = str(e).lower()
            if "429" in str(e) or "rate" in error_str:
                print(f"Rate limited. Waiting 30s before retry (attempt {attempt+1}/{max_retries})...")
                time.sleep(30)
            elif "402" in str(e) or "quota" in error_str or "exceeded" in error_str:
                print("API quota exceeded! Stopping.")
                return None  # Signal to stop processing
            elif "503" in str(e) or "loading" in error_str:
                print(f"Model loading. Waiting 20s...")
                time.sleep(20)
            else:
                print(f"HTTP Error: {e}")
                if attempt < max_retries - 1:
                    time.sleep(10)
                else:
                    return ["ERROR"] * len(query_names)
        except Exception as e:
            error_str = str(e).lower()
            if "429" in error_str or "rate" in error_str:
                print(f"Rate limited. Waiting 30s before retry (attempt {attempt+1}/{max_retries})...")
                time.sleep(30)
            elif "402" in error_str or "quota" in error_str or "exceeded" in error_str:
                print("API quota exceeded! Stopping.")
                return None  
            elif "503" in error_str or "loading" in error_str:
                print("Model loading. Waiting 20s...")
                time.sleep(20)
            else:
                print(f"Error: {e}")
                if attempt < max_retries - 1:
                    time.sleep(10)
                else:
                    return ["ERROR"] * len(query_names)
    
    return ["ERROR"] * len(query_names)

if len(no_matched_office_df) >= 2:
    test_names = no_matched_office_df['CompensationOffice1'].iloc[:2].tolist()
    print(f"Testing with:")
    for i, name in enumerate(test_names):
        print(f"  {i+1}. {name}")
    print()
    results = get_llm_match_batch(test_names, offices_list)
    if results:
        print("LLM suggested matches:")
        for i, result in enumerate(results):
            print(f"  {i+1}. {result}")
    else:
        print("API limit reached during test!")

Testing with:
  1. hess staatsministerium
  2. nan

HTTP Error: (Request ID: Root=1-69b1eb80-71adc530557736b071097d90;d9f9dd39-1eb3-4ef0-ba3c-5262937b9357)

Bad request:
HTTP Error: (Request ID: Root=1-69b1eb80-71adc530557736b071097d90;d9f9dd39-1eb3-4ef0-ba3c-5262937b9357)

Bad request:
HTTP Error: (Request ID: Root=1-69b1eb8b-327d3b150fb21f54500eaa1b;53e79627-7379-4e6b-b427-8c7f26fab5c4)

Bad request:
HTTP Error: (Request ID: Root=1-69b1eb8b-327d3b150fb21f54500eaa1b;53e79627-7379-4e6b-b427-8c7f26fab5c4)

Bad request:
HTTP Error: (Request ID: Root=1-69b1eb96-03491cd562a1abed5f8cafe4;88b979cc-beaa-4cfd-bf6d-8c5d12e7cb32)

Bad request:
LLM suggested matches:
  1. ERROR
  2. ERROR
HTTP Error: (Request ID: Root=1-69b1eb96-03491cd562a1abed5f8cafe4;88b979cc-beaa-4cfd-bf6d-8c5d12e7cb32)

Bad request:
LLM suggested matches:
  1. ERROR
  2. ERROR


In [ ]:
BATCH_SIZE = 10
DELAY_BETWEEN_BATCHES = 5  

output_file = 'hf_llm_matches_progress.jsonl'

processed = set()
llm_matches = []
try:
    with open(output_file, 'r') as f:
        for line in f:
            entry = json.loads(line)
            llm_matches.append(entry)
            processed.add(entry['CompensationOffice1'])
    print(f"Resuming from {len(processed)} already processed entries")
except FileNotFoundError:
    print("Starting fresh")

df_remaining = no_matched_office_df[~no_matched_office_df['CompensationOffice1'].isin(processed)]
print(f"Remaining to process: {len(df_remaining)}")

total_batches = (len(df_remaining) + BATCH_SIZE - 1) // BATCH_SIZE
api_limit_reached = False

for batch_idx in tqdm(range(total_batches), desc="Processing batches"):
    if api_limit_reached:
        break
        
    start_idx = batch_idx * BATCH_SIZE
    end_idx = min(start_idx + BATCH_SIZE, len(df_remaining))
    
    batch_df = df_remaining.iloc[start_idx:end_idx]
    batch_names = batch_df['CompensationOffice1'].tolist()
    
    batch_results = get_llm_match_batch(batch_names, offices_list)
    
    if batch_results is None:
        print("\nAPI limit reached! Saving progress and stopping.")
        api_limit_reached = True
        break
    
    for i, (idx, row) in enumerate(batch_df.iterrows()):
        entry = {
            'CompensationOffice1': row['CompensationOffice1'],
            'MatchedOffice': row['MatchedOffice'],
            'MatchDistance': float(row['MatchDistance']) if pd.notna(row['MatchDistance']) else None,
            'LLMMatch': batch_results[i] if i < len(batch_results) else "ERROR"
        }
        llm_matches.append(entry)
        
        with open(output_file, 'a') as f:
            f.write(json.dumps(entry) + '\n')
    
    if batch_idx < total_batches - 1 and not api_limit_reached:
        time.sleep(DELAY_BETWEEN_BATCHES)

df_hf_matches = pd.DataFrame(llm_matches)
print(f"\nProcessed: {len(df_hf_matches)} entries")
if api_limit_reached:
    print("Note: Processing stopped due to API limit. Run again later to continue.")
df_hf_matches.head(20)

In [ ]:
valid_matches = df_hf_matches[~df_hf_matches['LLMMatch'].isin(['NO_MATCH', 'ERROR', 'PARSE_ERROR', 'RATE_LIMITED'])]
no_matches = df_hf_matches[df_hf_matches['LLMMatch'] == 'NO_MATCH']
errors = df_hf_matches[df_hf_matches['LLMMatch'].isin(['ERROR', 'PARSE_ERROR', 'RATE_LIMITED'])]

print(f"Total processed: {len(df_hf_matches)}")
print(f"LLM found matches: {len(valid_matches)} ({len(valid_matches)/max(len(df_hf_matches),1)*100:.1f}%)")
print(f"No match found: {len(no_matches)}")
print(f"Errors: {len(errors)}")

df_hf_matches.to_excel('compensation_office_hf_llm_matches.xlsx', index=False)
print("\nSaved to compensation_office_hf_llm_matches.xlsx")